In [1]:
import os

from IPython.display import FileLink

if os.getcwd() == '/notebooks':
    os.chdir("./motion-synthesis")
    print('inside dir: ', os.listdir())

print("Click here to download the motion-dataset.zip: ", FileLink("motion-dataset.zip"))


inside dir:  ['requirements.txt', 'networks', 'dataset_processor.py', 'glove', 'exp_results', 'README.md', 'data_utils', 'checkpoints', '.git', 'options', 'main.ipynb', 'motion-dataset.zip', 'main.py', '.ipynb_checkpoints', 'data', '.gitignore', 'log', 'utils']
Click here to download the motion-dataset.zip:  /notebooks/motion-synthesis/motion-dataset.zip


In [2]:
import torch
from options.train_options  import TrainOptions
from os.path import join as pjoin
import os
from utils.paramUtils import t2m_kinematic_chain
import numpy as np
from utils.word_vectorizer import WordVectorizer
from torch.utils.data import DataLoader
from data_utils.dataset import MotionDatasetV2
from data_utils.dataset import PartMotionDatasetV2
from networks.nn import MotionVQVAE
from networks.trainers import MotionVQVAETrainer
from torch.utils.data import Subset


In [3]:
parser = TrainOptions()
options = parser.parse(args = ['--max_epoch', '5000'])
options.gpu_id = torch.cuda.current_device() if torch.cuda.is_available() else -1
options.device = torch.device("cpu" if options.gpu_id==-1 else "cuda:" + str(options.gpu_id))
torch.autograd.set_detect_anomaly(True)

if options.gpu_id != -1:
    # self.opt.gpu_id = int(self.opt.gpu_id)
    torch.cuda.set_device(options.gpu_id)

print('\nDevice used: ', options.device)

options.save_root = pjoin(options.checkpoints_dir, 'HumanML3D', options.name)
options.model_dir = pjoin(options.checkpoints_dir, 'model')
options.meta_dir = pjoin(options.save_root, 'meta')
options.eval_dir = pjoin(options.save_root, 'animation')
options.log_dir = pjoin('./log', options.dataset_name, options.name)
options.save_every_e = 100
options.is_continue = False

os.makedirs(options.model_dir, exist_ok=True)
os.makedirs(options.meta_dir, exist_ok=True)
os.makedirs(options.eval_dir, exist_ok=True)
os.makedirs(options.log_dir, exist_ok=True)

options.data_root = './data/HumanML3D'
options.motion_dir = pjoin(options.data_root, 'new_joint_vecs')
options.text_dir = pjoin(options.data_root, 'texts')
options.joints_num = 22
options.max_motion_length = 196
dim_pose = 263
radius = 4
fps = 20
kinematic_chain = t2m_kinematic_chain


Device used:  cuda:0


In [4]:
mean = np.load(pjoin(options.data_root, 'Mean.npy'))
std = np.load(pjoin(options.data_root, 'Std.npy'))

w_vectorizer = WordVectorizer('./glove', 'our_vab')
train_split_file = pjoin(options.data_root, 'train_micro.txt')
val_split_file = pjoin(options.data_root, 'val_micro.txt')

train_dataset = PartMotionDatasetV2(options, mean, std, train_split_file)
val_dataset = PartMotionDatasetV2(options, mean, std, val_split_file)
all_train_indices = np.arange(len(train_dataset))
micro_train_indices = all_train_indices[:80]
all_val_indices = np.arange(len(val_dataset))
micro_val_indices = all_val_indices[:30]
micro_train_dataset = Subset(train_dataset, micro_train_indices)
micro_val_dataset = Subset(val_dataset, micro_val_indices)

print('\nTrain Part dataset length: ', len(micro_train_dataset), len(train_dataset))
sample_motion = train_dataset[105]
print('Sample data shape: ', sample_motion['motion_parts'].shape)
Dp_max = sample_motion['motion_parts'].shape[-1]

id list 8


100%|██████████| 8/8 [00:00<00:00, 5424.25it/s]


Motion shape (B, T, D): (8, 199, 263)
Total number of motions 8, snippets 468
id list 4


100%|██████████| 4/4 [00:00<00:00, 3327.49it/s]

Motion shape (B, T, D): (4, 170, 263)
Total number of motions 4, snippets 495

Train Part dataset length:  80 468
Sample data shape:  (40, 6, 60)


In [5]:
train_loader = DataLoader(micro_train_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                              shuffle=False, pin_memory=True)
val_loader = DataLoader(micro_val_dataset, batch_size=options.batch_size, drop_last=not(options.dataset_mode == "micro"), num_workers=1,
                        shuffle=False, pin_memory=True)
vqvae = MotionVQVAE(
    input_dim=Dp_max,
    enc_hidden_dim=1024,
    dec_hidden_dim=1024,
    latent_dim=256,
    num_embeddings=512,
    beta=0.25
)

trainer = MotionVQVAETrainer(options, vqvae = vqvae)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader)


Number of epochs: 5000
Iters Per Epoch, Training: 0001, Validation: 001
Validation Loss: 0.61166 Reconstruction Loss: 0.41386 VQ Loss: 0.19780 Codebook Loss: 0.15824 Commitment Loss: 0.15824
epoch: 099 inner_iter:     0 0m 17s (- 14m 38s) niter: 0000100 completed:   2%) val_loss: 0.4509  loss: 0.9633  loss_rec: 0.8293  loss_vq: 0.1339  loss_codebook: 0.1072  loss_commit: 0.1072 
Validation Loss: 0.45081 Reconstruction Loss: 0.41913 VQ Loss: 0.03168 Codebook Loss: 0.02534 Commitment Loss: 0.02534
epoch: 199 inner_iter:     0 0m 35s (- 14m 5s) niter: 0000200 completed:   4%) val_loss: 0.4765  loss: 0.7640  loss_rec: 0.6505  loss_vq: 0.1135  loss_codebook: 0.0908  loss_commit: 0.0908 
Validation Loss: 0.47847 Reconstruction Loss: 0.41402 VQ Loss: 0.06445 Codebook Loss: 0.05156 Commitment Loss: 0.05156
epoch: 299 inner_iter:     0 0m 52s (- 13m 46s) niter: 0000300 completed:   6%) val_loss: 0.4369  loss: 0.5455  loss_rec: 0.3887  loss_vq: 0.1568  loss_codebook: 0.1255  loss_commit: 0.1255 